# Notebook 2



In [ ]:
# Imports
import numpy as np
import plotly.graph_objects as go
import math
from src.manifolds.sphere import Sphere

In [30]:
# Create a sphere object
sphere = Sphere()

# Chosen parameters
T = 1000 # Number of steps
dt = 0.01 #Time step of the simulation

# Choose a starting point on the sphere
point = np.array([1.0, 0.0, 0.0]) # On the sphere because unit length is 1

In [31]:
# Store trajectory of path after T number of steps in the simulation
path = [point.copy()] # Initialize array to contain the starting point
for i in range(T):
    point = sphere.euler_maruyama_step(point, dt)
    path.append(point.copy())
    
# Get x, y, z-coordinates for plotting
path = np.array(path) # Convert to numpy array to enable column indexing
x_coor = path[:, 0]
y_coor = path[:, 1]
z_coor = path[:, 2]

In [32]:
# Create sphere surface
# Get angle arrays
theta = np.linspace(0, 2*math.pi, num=500)
phi = np.linspace(0, math.pi, num=500)

# Turn theta and phi angles into a 2D grid
theta_grid, phi_grid = np.meshgrid(theta, phi)

# Apply sphere formulas for each axis of the surface
x_surface = np.sin(phi_grid) * np.cos(theta_grid)
y_surface = np.sin(phi_grid) * np.sin(theta_grid)
z_surface = np.cos(phi_grid)

# Create surface of sphere
surface = go.Surface(x=x_surface, y=y_surface, z=z_surface, opacity=0.5)

In [ ]:
# Create trace of trajectory for Brownian motion
motion_trace = go.Scatter3d(x=x_coor, y=y_coor, z=z_coor, mode="lines", line=dict(color="black"))

# Plot figure containing sphere surface and motion trace
figure = go.Figure(data=[surface, motion_trace])
figure.show()

In [34]:
# Verify all points stay on the sphere
# Go through each row of path to get norm of each point
norms = np.linalg.norm(path, axis=1)
tolerance = 1e-5 # Chosen tolerance for acceptable deviation
deviations = np.abs(norms-1)
assert deviations.all() <= tolerance
print(f"Maximum deviation from sphere: {deviations.max()}") # Print the maximum deviation from the unit length for a point in the path

Maximum deviation from sphere: 2.220446049250313e-16


In [ ]:
# Test idempotency of project_to_manifold method
point_on_sphere = np.array([0.0, 1.0, 0.0])
new_point = sphere.project_to_manifold(point_on_sphere)
second_new_point = sphere.project_to_manifold(new_point)
# Ensure both projections return the same point as the original
assert np.allclose(new_point, second_new_point, atol=1e-10)

# Test norm preservation after project_to_tangent method is applied
point_on_sphere = np.array([-1.0, 0.0, 0.0])
vector = np.array([1.0, 0.0, 0.0])
tangential_vector = sphere.project_to_tangent(point_on_sphere, vector)
assert abs(np.dot(point_on_sphere, tangential_vector)) < 1e-10

# Test that |X_n| = 1 after N steps
point_on_sphere = np.array([0.0, 0.0, -1.0])
N = 1000 # Chosen number of steps
dt = 0.01 # Chosen time step
tolerance = 1e-5 # Chosen tolerance
points = [point_on_sphere.copy()]

# Take N steps and add each new point to the list of points
for i in range(N):
   point = sphere.euler_maruyama_step(point, dt)
   points.append(point.copy())

points = np.array(points)
norms = np.linalg.norm(points, axis=1)
assert np.allclose(norms, 1.0, atol=tolerance)
    
# Test P_x^2 = P_x
point_on_sphere = np.array([0.0, -1.0, 0.0])
vector = np.array([0.0, 0.0, 1.0])
first_projection_vect = sphere.project_to_tangent(point_on_sphere, vector)
second_projection_vect = sphere.project_to_tangent(point_on_sphere, first_projection_vect)

tolerance = 1e-10 # Chosen tolerance
assert np.allclose(first_projection_vect, second_projection_vect, atol=tolerance)